# Extended Data Fig. 1: check of the effective Hamiltonian

Errors of the eigenvalues and eigenstates of the effective Hamiltonian
against the full one, for N = 12 random networks at phi = 1 (ferromagnetic
and ±J, 10000 networks per Gamma), normalized by 2^N.

Data: `data/hamapprox_N12_*.h5`
(`scripts/verify_hamiltonian_approximation.py`).

In [ ]:
%matplotlib inline

import glob

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from qising_networks import DATA_DIR as RESULTS
from qising_networks import FIGURES_DIR as FIGURES
from qising_networks import PROJECT_ROOT
from qising_networks.plots.common import COLOR_FERRO, COLOR_PMJ
from qising_networks.plots.figure import (
    MAX_FIG_WIDTH,
    align_panel_labels_to_yaxis,
    journal_labels,
    save_unlabeled,
)

if (PROJECT_ROOT / "matplotlibrc").exists():
    mpl.rc_file(PROJECT_ROOT / "matplotlibrc")

FIGURES.mkdir(exist_ok=True)
NUM_SPINS = 12
DIM = 2**NUM_SPINS
MODELS = ("ferro", "pmJ")
MODEL_COLOR = {"ferro": COLOR_FERRO, "pmJ": COLOR_PMJ}
MODEL_LABEL = {"ferro": "Ferromagnetic", "pmJ": r"$\pm J$"}

In [ ]:
def load_sweep(model, num_spins=NUM_SPINS):
    """Every Gamma's file of one model, as Gamma-sorted arrays."""
    pattern = str(RESULTS / f"hamapprox_N{num_spins}_{model}_gamma*.h5")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(pattern)
    gammas, eigval_mean, eigval_std, eigvec_mean, eigvec_std, nsamp = (
        [],
        [],
        [],
        [],
        [],
        [],
    )
    for fp in files:
        with h5py.File(fp) as f:
            p = dict(f["params"].attrs)
            ee = f["samples"]["eigval_error"][:] / DIM
            ve = f["samples"]["eigvec_error"][:] / DIM
        gammas.append(float(p["gamma"]))
        eigval_mean.append(ee.mean())
        eigval_std.append(ee.std())
        eigvec_mean.append(ve.mean())
        eigvec_std.append(ve.std())
        nsamp.append(len(ee))
    order = np.argsort(gammas)
    return {
        "gamma": np.array(gammas)[order],
        "eigval_mean": np.array(eigval_mean)[order],
        "eigval_std": np.array(eigval_std)[order],
        "eigvec_mean": np.array(eigvec_mean)[order],
        "eigvec_std": np.array(eigvec_std)[order],
        "nsamples": np.array(nsamp)[order],
    }


sweeps = {model: load_sweep(model) for model in MODELS}
for model, s in sweeps.items():
    print(model, "gamma:", s["gamma"], "nsamples:", s["nsamples"])

In [ ]:
# Reference Gamma^2 line through each panel's ferromagnetic smallest-Gamma
# point (second-order perturbation theory fixes the power, not the
# prefactor).
gamma_ref = np.linspace(
    sweeps["ferro"]["gamma"].min(), sweeps["ferro"]["gamma"].max(), 100
)

fig = plt.figure(figsize=(MAX_FIG_WIDTH, 2.7))
axes = fig.subplots(1, 2)

panel_defs = (
    (
        "eigval_mean",
        "eigval_std",
        r"Eigenenergy error $\overline{|\Delta E|}/J$",
    ),
    (
        "eigvec_mean",
        "eigvec_std",
        r"Eigenstate error $\overline{1-\langle \hat{P} \rangle}$",
    ),
)
for ax, (mkey, skey, ylabel) in zip(axes, panel_defs):
    g0, e0 = sweeps["ferro"]["gamma"][0], sweeps["ferro"][mkey][0]
    quad_ref = e0 * (gamma_ref / g0) ** 2
    ax.loglog(
        gamma_ref,
        quad_ref,
        ls=":",
        lw=0.9,
        color="0.6",
        zorder=1,
        label=r"$\propto (\Gamma/J)^2$",
    )
    for model in MODELS:
        s = sweeps[model]
        ax.errorbar(
            s["gamma"],
            s[mkey],
            yerr=s[skey] / np.sqrt(s["nsamples"]),
            fmt="o-",
            ms=3.5,
            lw=1.0,
            capsize=1.5,
            color=MODEL_COLOR[model],
            label=MODEL_LABEL[model],
            zorder=3,
        )
    ax.set_xlabel(r"Transverse field $\Gamma/J$")
    ax.set_ylabel(ylabel)
    ax.set_xlim(0.08, 1.0)

axes[0].legend(frameon=False, loc="upper left")

labels = journal_labels("ab")
align_panel_labels_to_yaxis(fig, axes, labels)

In [ ]:
paths = save_unlabeled(fig, FIGURES / "fig-s01", "fig-s01")
print(paths)